# ⚡ Lección 2 – Apache Spark: Introducción y Configuración
### Proyecto: Retail Analytics Pipeline — RetailMax
**Módulo 9: Fundamentos de Big Data | Alkemy**

---
**Objetivo:** Configurar Spark e iniciar el procesamiento distribuido del dataset Fashion-MNIST.

## 1. Instalación y verificación del entorno

**Requisitos previos:**
- Python 3.8+
- Java 8 o 11 (necesario para Spark)
- PySpark (`pip install pyspark`)

**Verificar Java instalado:** Ejecutar en terminal: `java -version`

In [6]:
# ============================================================
# Verificación del entorno antes de iniciar Spark
# ============================================================
import sys, os, subprocess

print('=== VERIFICACIÓN DEL ENTORNO ===')
print(f'Python version : {sys.version}')

try:
    java_ver = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT).decode()
    print(f'Java           : {java_ver.strip().splitlines()[0]}')
except Exception as e:
    print(f'Java           : NO ENCONTRADO — Instalar JDK 11: https://adoptium.net/')

try:
    import pyspark
    print(f'PySpark version: {pyspark.__version__}')
except ImportError:
    print('PySpark        : NO INSTALADO — ejecutar: pip install pyspark')

print('\n✅ Si ves versiones de Python, Java y PySpark, el entorno está listo.')

=== VERIFICACIÓN DEL ENTORNO ===
Python version : 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Java           : openjdk version "11.0.30" 2026-01-20
PySpark version: 4.1.1

✅ Si ves versiones de Python, Java y PySpark, el entorno está listo.


In [7]:
# ============================================================
# Configuración de SparkSession (modo ultra-ligero)
# ============================================================
from pyspark.sql import SparkSession

# Configuración minimalista para evitar demoras
spark = (
    SparkSession.builder
    .appName('RetailMax_FashionMNIST')
    .master('local[1]')                          # Solo 1 núcleo
    .config('spark.driver.memory', '512m')       # Memoria mínima
    .config('spark.executor.memory', '512m')     # Memoria mínima
    .config('spark.sql.shuffle.partitions', '2') # Mínimas particiones
    .config('spark.ui.enabled', 'false')         # Deshabilitar UI web
    .config('spark.sql.adaptive.enabled', 'false') # Deshabilitar optimizaciones
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel('ERROR')  # Solo errores críticos

print('=== SPARK ULTRA-LIGERO CONFIGURADO ===')
print(f'App Name: {sc.appName}')
print(f'Master  : {sc.master}')
print(f'Versión : {sc.version}')
print('✅ Spark listo (si ves este mensaje, inició correctamente)')

KeyboardInterrupt: 

## 2. SparkContext vs SparkSession

| Concepto | Descripción | Cuándo usar |
|----------|-------------|-------------|
| **SparkContext (sc)** | Conexión al cluster Spark. API original para RDDs | RDDs de bajo nivel |
| **SparkSession (spark)** | Punto de entrada unificado (Spark 2+). Incluye sc internamente | DataFrames, SQL, MLlib |

> 💡 En Spark 3.x se recomienda siempre usar `SparkSession` y acceder al `SparkContext` vía `spark.sparkContext`.

In [ ]:
# ============================================================
# Carga inicial del dataset Fashion-MNIST en un RDD
# ============================================================
import os

DATA_PATH = 'fashion_mnist'
TRAIN_CSV = os.path.join(DATA_PATH, 'fashion_train.csv')
TEST_CSV  = os.path.join(DATA_PATH, 'fashion_test.csv')

# Cargar como RDD de texto (cada línea es un string)
rdd_raw_train = sc.textFile(TRAIN_CSV)
rdd_raw_test  = sc.textFile(TEST_CSV)

print('=== CARGA EN RDD ===')
print(f'Particiones (train): {rdd_raw_train.getNumPartitions()}')
print(f'Particiones (test) : {rdd_raw_test.getNumPartitions()}')

# Acción: count — dispara la ejecución real
total_train = rdd_raw_train.count()
total_test  = rdd_raw_test.count()

print(f'\nTotal filas train (incluye header): {total_train:,}')
print(f'Total filas test  (incluye header): {total_test:,}')

In [ ]:
# ============================================================
# Explorar primeras filas con la acción take()
# ============================================================
print('=== PRIMERAS 3 FILAS DEL RDD (RAW) ===')
for i, row in enumerate(rdd_raw_train.take(3)):
    preview = row[:120] + '...' if len(row) > 120 else row
    print(f'[{i}]: {preview}')

# Separar header
header = rdd_raw_train.first()
cols = header.split(',')
print(f'\n=== CABECERA (primeras 5 columnas) ===')
print(cols[:5], '...')
print(f'Total columnas: {len(cols)}')

In [ ]:
# ============================================================
# RDD limpio: sin header, parseado a lista de valores
# ============================================================
header_train = rdd_raw_train.first()
header_test  = rdd_raw_test.first()

# Filtrar header y parsear valores
rdd_train = (
    rdd_raw_train
    .filter(lambda line: line != header_train)  # Eliminar header
    .map(lambda line: line.split(','))           # Dividir por coma
    .map(lambda fields: (
        int(fields[0]),                          # label (int)
        fields[1],                               # label_name (str)
        [int(x) for x in fields[2:]]            # pixels (list of int)
    ))
)

rdd_test = (
    rdd_raw_test
    .filter(lambda line: line != header_test)
    .map(lambda line: line.split(','))
    .map(lambda fields: (
        int(fields[0]),
        fields[1],
        [int(x) for x in fields[2:]]
    ))
)

# Persistir en memoria para reutilización eficiente
rdd_train.cache()
rdd_test.cache()

print('=== RDD PARSEADO Y EN CACHÉ ===')
print(f'Train count: {rdd_train.count():,}')
print(f'Test  count: {rdd_test.count():,}')
print(f'\nEjemplo de un registro (label, label_name, pixels[:5]):')
sample = rdd_train.take(1)[0]
print(f'  label      : {sample[0]}')
print(f'  label_name : {sample[1]}')
print(f'  pixels[:5] : {sample[2][:5]}')
print(f'  total pixels: {len(sample[2])}')

In [ ]:
# ============================================================
# Validación de conectividad y acciones básicas
# ============================================================
import numpy as np

print('=== ACCIONES BÁSICAS SOBRE EL RDD ===')

# count: total registros
n_train = rdd_train.count()
print(f'count()  train : {n_train:,}')

# take: primeros N registros
primeros = rdd_train.take(5)
print(f'take(5)        : {[(r[0], r[1]) for r in primeros]}')

# first: primer registro
primero = rdd_train.first()
print(f'first()  label : {primero[0]} ({primero[1]})')

# Distribución de clases
dist_clases = (
    rdd_train
    .map(lambda r: (r[1], 1))
    .reduceByKey(lambda a, b: a + b)
    .sortByKey()
    .collect()
)
print(f'\nDistribución de clases:')
for nombre, cnt in dist_clases:
    bar = '█' * (cnt // 500)
    print(f'  {nombre:<15} {cnt:>6,}  {bar}')

print('\n✅ Conectividad con el dataset validada correctamente.')

In [ ]:
# ============================================================
# Guardar contexto para Lección 3
# ============================================================
import pickle, os

# Los RDDs se regenerarán en L3 a partir del CSV
# Guardamos rutas y configuraciones
config = {
    'train_csv': TRAIN_CSV,
    'test_csv' : TEST_CSV,
    'n_train'  : rdd_train.count(),
    'n_test'   : rdd_test.count(),
    'n_classes': 10
}

os.makedirs('outputs', exist_ok=True)
with open('outputs/L2_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print('✅ Configuración guardada en outputs/L2_config.pkl')
print('\n📌 Lección 2 completada. Próximo paso → Lección 3: RDD, Transformaciones y Acciones')

---
## ✅ Checklist Lección 2
- [x] SparkContext y SparkSession configurados en modo local
- [x] Dataset Fashion-MNIST cargado en RDD
- [x] Acciones básicas ejecutadas: `count()`, `take()`, `first()`
- [x] Conectividad con el dataset validada
- [x] RDDs persistidos en caché para eficiencia

**Próximo paso → Lección 3:** Transformaciones avanzadas con RDDs y Pair RDDs.